# Clase 218 — Content-based: TF-IDF + sentence-transformers + FAISS

Movies sintéticas con título + overview + genres. Recomendamos por similitud de contenido.

Requiere: `pip install scikit-learn sentence-transformers faiss-cpu`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.DataFrame([
    {'id': 1,  'title': 'Toy Story',           'overview': 'Cowboy doll and astronaut toy adventures friendship.', 'genres': 'animation children comedy'},
    {'id': 2,  'title': 'Jumanji',             'overview': 'Magical board game jungle wild animals.',              'genres': 'adventure children fantasy'},
    {'id': 3,  'title': 'Heat',                'overview': 'Bank heist crew detective pursuit Los Angeles.',       'genres': 'action crime thriller'},
    {'id': 4,  'title': 'Goldeneye',           'overview': 'British spy secret agent satellite weapon villain.',   'genres': 'action adventure thriller'},
    {'id': 5,  'title': 'The Lion King',       'overview': 'Lion cub father betrayed uncle Africa savanna.',       'genres': 'animation adventure drama'},
    {'id': 6,  'title': 'Pulp Fiction',        'overview': 'Hitmen boxer gangster intersecting stories LA.',       'genres': 'crime drama thriller'},
    {'id': 7,  'title': 'Finding Nemo',        'overview': 'Clownfish ocean adventure son rescue father.',         'genres': 'animation children adventure'},
    {'id': 8,  'title': 'The Matrix',          'overview': 'Hacker discovers reality simulation rebel against AI.',
                                                                                                                     'genres': 'action sci-fi thriller'},
    {'id': 9,  'title': 'Forrest Gump',        'overview': 'Slow-witted man witnesses 20th century history love.', 'genres': 'comedy drama romance'},
    {'id': 10, 'title': 'Inception',           'overview': 'Dream thief mind heist subconscious layers.',          'genres': 'action sci-fi thriller'},
    {'id': 11, 'title': 'Shrek',               'overview': 'Ogre swamp princess rescue donkey fairy tale.',        'genres': 'animation comedy adventure'},
    {'id': 12, 'title': 'The Dark Knight',     'overview': 'Batman Joker chaos Gotham villain moral.',             'genres': 'action crime thriller'},
])

movies['text'] = movies.title + ' ' + movies.overview + ' ' + movies.genres
print(movies[['id', 'title']].to_string(index=False))

## 1. TF-IDF + item similarity

In [ ]:
vec = TfidfVectorizer(max_features=200, ngram_range=(1, 2), stop_words='english')
M = vec.fit_transform(movies.text)
print(f'TF-IDF shape: {M.shape}')

sim = cosine_similarity(M)

def top_similar_tfidf(title, n=5):
    i = movies.index[movies.title == title][0]
    scores = sim[i].copy(); scores[i] = -1
    top = np.argsort(-scores)[:n]
    return [(movies.iloc[j].title, round(scores[j], 3)) for j in top]

for t in ['Toy Story', 'Heat', 'The Matrix']:
    print(f'\nSimilar a "{t}":')
    for name, s in top_similar_tfidf(t): print(f'  {s:.3f}  {name}')

## 2. User profile + recomendación

In [ ]:
# user_42 le gustaron 3 animaciones (ratings ≥ 4)
user_ratings = {1: 5, 5: 4, 7: 5}   # Toy Story, Lion King, Finding Nemo

from scipy.sparse import csr_matrix
weights = np.array([user_ratings.get(mid, 0) for mid in movies.id])
mask = weights > 0
user_profile = (M[mask].multiply(weights[mask][:, None])).sum(axis=0) / weights[mask].sum()
user_profile = np.asarray(user_profile).ravel()

scores = M @ user_profile
# Excluir las que ya vio
scores[mask] = -1
top = np.argsort(-scores)[:5]
print('Recomendaciones para user que le gustaron Toy Story, Lion King, Finding Nemo:')
for j in top:
    print(f'  {scores[j]:.3f}  {movies.iloc[j].title}  ({movies.iloc[j].genres})')

## 3. Embeddings semánticos con sentence-transformers

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer('all-MiniLM-L6-v2')
    emb = st.encode(movies.text.tolist(), normalize_embeddings=True)
    print('embeddings shape:', emb.shape)

    def top_similar_emb(title, n=5):
        i = movies.index[movies.title == title][0]
        s = emb @ emb[i]; s[i] = -1
        top = np.argsort(-s)[:n]
        return [(movies.iloc[j].title, round(float(s[j]), 3)) for j in top]

    for t in ['Toy Story', 'The Matrix']:
        print(f'\nSimilar a "{t}" (embeddings):')
        for name, s in top_similar_emb(t): print(f'  {s:.3f}  {name}')
        print(f'  vs TF-IDF:')
        for name, s in top_similar_tfidf(t): print(f'  {s:.3f}  {name}')
except ImportError:
    print('pip install sentence-transformers para esta celda')

## 4. FAISS para retrieval rápido (escala a millones)

In [ ]:
try:
    import faiss, time
    d = emb.shape[1]
    index = faiss.IndexFlatIP(d)   # inner product (= cosine si normalizamos)
    index.add(emb.astype('float32'))
    print(f'index size: {index.ntotal} items')

    user_profile_emb = emb[[0, 4, 6]].mean(axis=0)   # gusto = avg de Toy Story + Lion King + Nemo
    user_profile_emb = user_profile_emb / np.linalg.norm(user_profile_emb)

    t0 = time.perf_counter()
    D, I = index.search(user_profile_emb.astype('float32').reshape(1, -1), k=5)
    print(f'FAISS top-5 search: {(time.perf_counter() - t0) * 1000:.2f} ms')
    for j, s in zip(I[0], D[0]):
        print(f'  {s:.3f}  {movies.iloc[j].title}')
except (ImportError, NameError):
    print('pip install faiss-cpu (y sentence-transformers para emb) para esta celda')

## Ejercicio guiado

1. Bajá MovieLens + sinopsis (TMDB API o Kaggle). Repetí con 10K movies reales.
2. Comparé top-10 TF-IDF vs sentence-transformers vs CF (Clase 216) para el mismo user.
3. Coverage: ¿qué % del catálogo es recomendado al menos una vez para 1000 users distintos?
4. MMR (Maximal Marginal Relevance) para diversidad: penalizar items similares a los ya seleccionados.
5. Cold-start demo: agregá una movie nueva (sin ratings). CF la ignora; content-based la recomienda.

## Conclusiones

- Content-based brilla en cold-start de items y dominios con metadata rica.
- TF-IDF sigue siendo baseline sólido; embeddings añaden semántica.
- FAISS hace retrieval factible a escala de millones de items.
- Solo content-based = filter bubble; combinar con CF (Clase 219).